# Pipeline walkthrough: Port Boca Grande king tide alerts

This notebook runs the same pipeline as `transform.py` and `alerts.py`, one step at a time, and shows the table that comes out of each step. Run the cells in order from top to bottom, because each cell builds on the ones above it.

It never calls NOAA. It reads the raw files already committed in `data/raw/`.

## Before you start

1. Save this notebook in the root of the `de-application` repository, next to `transform.py`.
2. In a terminal in that folder, turn on the virtual environment and install Jupyter and pandas:
   ```
   pip install jupyterlab pandas
   ```
3. Start Jupyter with `jupyter lab`, then open this notebook.

**How to read the outputs:** every query result appears as a pandas DataFrame, the same kind of table you'd see in a data science notebook. A test that passes shows an empty table.

## 0. Set up

This cell imports the libraries, makes sure the notebook is working from the repository root (the SQL files use paths like `data/raw/...`), and opens a database.

The database is **in memory** (`duckdb.connect()` with no file name). It disappears when you restart the kernel. That's deliberate: DuckDB lets only one program write to a database file at a time, so if this notebook held `tides.duckdb` open, `python transform.py` would fail in your terminal until you shut the notebook down.

In [1]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd

# Find the repository root: the folder that contains transform.py.
# This lets the notebook work from the root or from a subfolder like notebooks/.
here = Path.cwd()
for folder in [here, *here.parents]:
    if (folder / "transform.py").exists():
        os.chdir(folder)
        break
print("Working folder:", Path.cwd())

# An in-memory database. Nothing is written to disk.
con = duckdb.connect()

# Show more rows and columns than pandas does by default.
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 30)

def q(sql):
    """Run a query and return the result as a pandas DataFrame."""
    return con.sql(sql).df()

def show_file(path):
    """Print a SQL file, so you can read the real code before running it."""
    print(Path(path).read_text(encoding="utf-8"))

def run_file(path):
    """Run a SQL file from the repository, exactly as transform.py does."""
    con.sql(Path(path).read_text(encoding="utf-8"))
    print(f"Ran {path}")

Working folder: C:\Users\SalomeGrasland\Desktop\2026\DEApp\de-application


## 1. Raw data: what NOAA sent

`extract.py` saved three responses under `data/raw/`, one subfolder per request. Each file name holds the station ID and the UTC time of the pull.

In [33]:
import requests
import extract

class StopBeforeSending(Exception):
    """Raised by the stand-in below, so no request reaches NOAA."""

captured_urls = []

def record_instead_of_sending(url, params=None, timeout=None):
    # Build the full URL the same way requests would, including the
    # ?key=value parameters, then stop before sending it.
    full_url = requests.Request("GET", url, params=params).prepare().url
    captured_urls.append(full_url)
    raise StopBeforeSending

real_get = requests.get
requests.get = record_instead_of_sending      # swap in the stand-in
try:
    # extract.STEPS is the real list of (subfolder, fetch function) pairs.
    for subfolder, fetch in extract.STEPS:
        try:
            fetch()
        except StopBeforeSending:
            print(f"{subfolder}:\n  {captured_urls[-1]}\n")
finally:
    requests.get = real_get                   # always put the real one back

stations:
  https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/8725577.json

predictions:
  https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?product=predictions&station=8725577&begin_date=20260801&end_date=20270731&datum=MLLW&interval=hilo&units=english&time_zone=gmt&format=json&application=de_application

datums:
  https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/8725577/datums.json



In [2]:
for path in sorted(Path("data/raw").rglob("*.json")):
    print(path, f"({path.stat().st_size:,} bytes)")

data\raw\datums\8725577_2026-11-17T194804Z.json (2,171 bytes)
data\raw\predictions\8725577_2026-11-17T194803Z.json (55,130 bytes)
data\raw\stations\8725577_2026-11-18T164914Z.json (2,724 bytes)


### Predictions

The file is one JSON object with a single key, `predictions`, holding a list. Each entry is one tide. Look at the values: every one is **text**, including the height (`"v"`) and the time (`"t"`). Staging fixes that.

In [3]:
pred_file = sorted(Path("data/raw/predictions").glob("*.json"))[-1]
raw_pred = json.loads(pred_file.read_text(encoding="utf-8"))

print("Top-level keys:", list(raw_pred.keys()))
print("Number of tides:", len(raw_pred["predictions"]))
print("First five entries:")
for entry in raw_pred["predictions"][:5]:
    print(" ", entry)

Top-level keys: ['predictions']
Number of tides: 1101
First five entries:
  {'t': '2026-08-01 01:14', 'v': '0.116', 'type': 'L'}
  {'t': '2026-08-01 08:07', 'v': '1.144', 'type': 'H'}
  {'t': '2026-08-01 12:22', 'v': '0.807', 'type': 'L'}
  {'t': '2026-08-01 18:21', 'v': '1.621', 'type': 'H'}
  {'t': '2026-08-02 01:38', 'v': '0.276', 'type': 'L'}


### Datums

This response has many keys. The pipeline uses `units`, `epoch`, `ctrlStation`, and the `datums` list. Each entry in `datums` is a name and a value.

In [4]:
datum_file = sorted(Path("data/raw/datums").glob("*.json"))[-1]
raw_datums = json.loads(datum_file.read_text(encoding="utf-8"))

print("units:", raw_datums["units"])
print("epoch:", raw_datums["epoch"])
print("ctrlStation:", raw_datums["ctrlStation"])
pd.DataFrame(raw_datums["datums"])

units: feet
epoch: 1983-2001
ctrlStation: 8725110 Naples, FL


,name,description,value
0,STND,Station Datum,0.00
1,MHHW,Mean Higher-High Water,3.38
2,MHW,Mean High Water,3.15
3,DTL,Mean Diurnal Tide Level,2.60
4,MTL,Mean Tide Level,2.68
5,MSL,Mean Sea Level,2.69
6,MLW,Mean Low Water,2.21
7,MLLW,Mean Lower-Low Water,1.82
8,GT,Great Diurnal Range,1.56
9,MN,Mean Range of Tide,0.93


### Stations

The station response is a list called `stations` with one entry, because `extract.py` asked for one station. Only a few of its fields reach the pipeline.

In [5]:
station_file = sorted(Path("data/raw/stations").glob("*.json"))[-1]
raw_station = json.loads(station_file.read_text(encoding="utf-8"))

station = raw_station["stations"][0]
{key: station[key] for key in ["id", "name", "tideType", "lat", "lng"]}

{'id': '8725577',
 'name': 'PORT BOCA GRANDE, CHARLOTTE HBR.',
 'tideType': 'Mixed Diurnal',
 'lat': 26.72,
 'lng': -82.2583}

## 2. Staging: `stg_predictions`

The SQL file builds this table in three steps, written as CTEs (named steps inside one query, defined with `with ... as`). To see each step's output, the next cells rebuild the steps as **temporary views**: saved queries you can select from. Each view reads from the one before it, so you can watch the data change shape.

These views are copies for viewing only. The real table still comes from `sql/staging/stg_predictions.sql`, a few cells down.

### Step 1a: what `read_json` sees

Before unnesting, DuckDB reads each file as **one row**, with the whole list of tides packed into one column.

In [6]:
q("""
    select
        filename,
        len(predictions) as tides_in_list
    from read_json('data/raw/predictions/*.json', filename = true)
""")

,filename,tides_in_list
0,data/raw/predictions/8725577_2026-11-17T194803...,1101


### Step 1b: `unnest` turns the list into rows

`unnest(..., recursive := true)` makes one row per tide and splits each entry into its own columns: `t`, `v`, and `type`. Check the `dtypes` printout under the table: everything is still text (`object` in pandas).

In [7]:
con.sql("""
    create or replace temp view peek_pred_unnested as
    select
        filename,
        unnest(predictions, recursive := true)
    from read_json('data/raw/predictions/*.json', filename = true)
""")

df = q("select * from peek_pred_unnested limit 8")
display(df)
print(df.dtypes)

,filename,t,v,type
0,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-01 01:14,0.116,L
1,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-01 08:07,1.144,H
2,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-01 12:22,0.807,L
3,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-01 18:21,1.621,H
4,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-02 01:38,0.276,L
5,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-02 08:23,1.225,H
6,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-02 13:21,0.723,L
7,data/raw/predictions/8725577_2026-11-17T194803...,2026-08-02 19:09,1.42,H


filename    str
t           str
v           str
type        str
dtype: object


### Step 2: give every column a proper type

This step pulls the station ID and pull time out of the file name, turns `t` into a timestamp, `v` into a number, and `H`/`L` into `high`/`low`. Compare the `dtypes` with the cell above.

In [8]:
con.sql(r"""
    create or replace temp view peek_pred_typed as
    select
        regexp_extract(filename, '(\d{7})_', 1) as station_id,
        strptime(regexp_extract(filename, '_(\d{4}-\d{2}-\d{2}T\d{6}Z)', 1), '%Y-%m-%dT%H%M%SZ') as pulled_at_utc,
        strptime(t, '%Y-%m-%d %H:%M') as tide_at_utc,
        cast(v as double) as height_ft_mllw,
        case type when 'H' then 'high' when 'L' then 'low' end as tide_type
    from peek_pred_unnested
""")

df = q("select * from peek_pred_typed limit 8")
display(df)
print(df.dtypes)

,station_id,pulled_at_utc,tide_at_utc,height_ft_mllw,tide_type
0,8725577,2026-11-17 19:48:03,2026-08-01 01:14:00,0.116,low
1,8725577,2026-11-17 19:48:03,2026-08-01 08:07:00,1.144,high
2,8725577,2026-11-17 19:48:03,2026-08-01 12:22:00,0.807,low
3,8725577,2026-11-17 19:48:03,2026-08-01 18:21:00,1.621,high
4,8725577,2026-11-17 19:48:03,2026-08-02 01:38:00,0.276,low
5,8725577,2026-11-17 19:48:03,2026-08-02 08:23:00,1.225,high
6,8725577,2026-11-17 19:48:03,2026-08-02 13:21:00,0.723,low
7,8725577,2026-11-17 19:48:03,2026-08-02 19:09:00,1.420,high


station_id                   str
pulled_at_utc     datetime64[us]
tide_at_utc       datetime64[us]
height_ft_mllw           float64
tide_type                    str
dtype: object


### Step 3: remove duplicates

`row_number()` numbers the copies of each tide, newest pull first, and `qualify ... = 1` keeps only row 1. With one raw file there are no duplicates yet, so to see the rule work, this cell makes a fake second pull: every tide copied with a later `pulled_at_utc`.

Look at `row_num`. Each tide now has a 1 (the newer copy) and a 2 (the older one), and only the 1s survive.

In [9]:
q("""
    with two_pulls as (
        select * from peek_pred_typed
        union all
        select * replace (pulled_at_utc + interval 1 day as pulled_at_utc)
        from peek_pred_typed
    )
    select
        *,
        row_number() over (
            partition by station_id, tide_at_utc
            order by pulled_at_utc desc
        ) as row_num
    from two_pulls
    order by tide_at_utc, row_num
    limit 6
""")

,station_id,pulled_at_utc,tide_at_utc,height_ft_mllw,tide_type,row_num
0,8725577,2026-11-18 19:48:03,2026-08-01 01:14:00,0.116,low,1
1,8725577,2026-11-17 19:48:03,2026-08-01 01:14:00,0.116,low,2
2,8725577,2026-11-18 19:48:03,2026-08-01 08:07:00,1.144,high,1
3,8725577,2026-11-17 19:48:03,2026-08-01 08:07:00,1.144,high,2
4,8725577,2026-11-18 19:48:03,2026-08-01 12:22:00,0.807,low,1
5,8725577,2026-11-17 19:48:03,2026-08-01 12:22:00,0.807,low,2


### Build the real table

Now read the actual file and run it. The output should match Step 2, with duplicates removed.

In [10]:
show_file("sql/staging/stg_predictions.sql")

-- stg_predictions.sql
-- Staging table for tide predictions: one clean row per tide.
-- Reads every raw predictions file, fixes the types, and removes duplicate
-- tides, keeping the copy from the most recent pull.
-- "create or replace" means each run rebuilds this table from scratch,
-- so running it twice never adds rows.

create or replace table stg_predictions as

-- Step 1: read the raw files and flatten them.
with unnested as (
    select
        -- filename = true (below) adds this column: the path of the file each
        -- row came from. The station ID and pull time are only in the file
        -- name, not in NOAA's response, so this column is needed later.
        filename,
        -- Each file holds one list called "predictions", with one entry per
        -- tide: t (time), v (height), and type (H or L).
        -- unnest turns the list into one row per tide, and recursive := true
        -- splits each entry into its own columns: t, v, and type.
        unnest(predicti

In [11]:
run_file("sql/staging/stg_predictions.sql")

display(q("select * from stg_predictions order by tide_at_utc limit 10"))
q("""
    select tide_type, count(*) as tides, min(tide_at_utc) as first_tide, max(tide_at_utc) as last_tide
    from stg_predictions
    group by tide_type
""")

Ran sql/staging/stg_predictions.sql


,station_id,pulled_at_utc,tide_at_utc,height_ft_mllw,tide_type
0,8725577,2026-11-17 19:48:03,2026-08-01 01:14:00,0.116,low
1,8725577,2026-11-17 19:48:03,2026-08-01 08:07:00,1.144,high
2,8725577,2026-11-17 19:48:03,2026-08-01 12:22:00,0.807,low
3,8725577,2026-11-17 19:48:03,2026-08-01 18:21:00,1.621,high
4,8725577,2026-11-17 19:48:03,2026-08-02 01:38:00,0.276,low
5,8725577,2026-11-17 19:48:03,2026-08-02 08:23:00,1.225,high
6,8725577,2026-11-17 19:48:03,2026-08-02 13:21:00,0.723,low
7,8725577,2026-11-17 19:48:03,2026-08-02 19:09:00,1.420,high
8,8725577,2026-11-17 19:48:03,2026-08-03 01:57:00,0.463,low
9,8725577,2026-11-17 19:48:03,2026-08-03 08:32:00,1.337,high


,tide_type,tides,first_tide,last_tide
0,high,550,2026-08-01 08:07:00,2027-07-31 14:46:00
1,low,551,2026-08-01 01:14:00,2027-07-31 22:53:00


## 3. Staging: `stg_datums`

This table goes from **long** (one row per datum) to **wide** (one row per station), the same move as a Cross Tab in Alteryx.

### Before: one row per datum

In [12]:
con.sql(r"""
    create or replace temp view peek_datums_long as
    select
        regexp_extract(filename, '(\d{7})_', 1) as station_id,
        units,
        epoch,
        ctrlStation as control_station,
        unnest(datums, recursive := true)
    from read_json('data/raw/datums/*.json', filename = true)
""")

q("select * from peek_datums_long")

,station_id,units,epoch,control_station,name,description,value
0,8725577,feet,1983-2001,"8725110 Naples, FL",STND,Station Datum,0.00
1,8725577,feet,1983-2001,"8725110 Naples, FL",MHHW,Mean Higher-High Water,3.38
2,8725577,feet,1983-2001,"8725110 Naples, FL",MHW,Mean High Water,3.15
3,8725577,feet,1983-2001,"8725110 Naples, FL",DTL,Mean Diurnal Tide Level,2.60
4,8725577,feet,1983-2001,"8725110 Naples, FL",MTL,Mean Tide Level,2.68
5,8725577,feet,1983-2001,"8725110 Naples, FL",MSL,Mean Sea Level,2.69
6,8725577,feet,1983-2001,"8725110 Naples, FL",MLW,Mean Low Water,2.21
7,8725577,feet,1983-2001,"8725110 Naples, FL",MLLW,Mean Lower-Low Water,1.82
8,8725577,feet,1983-2001,"8725110 Naples, FL",GT,Great Diurnal Range,1.56
9,8725577,feet,1983-2001,"8725110 Naples, FL",MN,Mean Range of Tide,0.93


### After: one row per station

Each `max(case when name = 'MHHW' then value end)` picks one value out of the long rows. The last column, `mhhw_above_mllw_ft`, is the 1.56-foot gap the mart subtracts from every tide.

In [13]:
show_file("sql/staging/stg_datums.sql")

-- stg_datums.sql
-- Staging table for tidal datums: one row per station.
-- A datum is a reference water level that NOAA averages over many years.
-- This table keeps two of them, MHHW (the average higher high tide) and
-- MLLW (the average lower low tide), and the gap between them.
-- The predictions measure heights from MLLW, so the marts subtract this gap
-- to get each tide's height above a normal high tide.
-- "create or replace" means each run rebuilds this table from scratch,
-- so running it twice never adds rows.

create or replace table stg_datums as

-- Step 1: read the raw files and flatten them.
with unnested as (
    select
        -- The station ID: the 7 digits before the underscore in the file name.
        -- 8725577_2026-11-17T194804Z.json -> 8725577
        regexp_extract(filename, '(\d{7})_', 1) as station_id,

        -- The pull time: the timestamp after the underscore, as a real timestamp.
        -- Used below to pick the newest file.
        -- 8725577_2026-1

In [14]:
run_file("sql/staging/stg_datums.sql")
q("select * from stg_datums")

Ran sql/staging/stg_datums.sql


,station_id,units,datum_epoch,control_station,mhhw_ft_stnd,mllw_ft_stnd,mhhw_above_mllw_ft
0,8725577,feet,1983-2001,"8725110 Naples, FL",3.38,1.82,1.56


## 4. Staging: `stg_stations`

One row per station, with clearer column names. `tideType` becomes `tide_pattern` so it isn't confused with `tide_type` (high or low) in `stg_predictions`.

In [15]:
run_file("sql/staging/stg_stations.sql")
q("select * from stg_stations")

Ran sql/staging/stg_stations.sql


,station_id,station_name,tide_pattern,latitude,longitude
0,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",Mixed Diurnal,26.72,-82.2583


### Check all three staging tables

`describe` lists each column and its type, the same as checking field types in Alteryx's Select tool.

In [16]:
for table in ["stg_predictions", "stg_datums", "stg_stations"]:
    print(f"\n{table}")
    display(q(f"describe {table}")[["column_name", "column_type"]])


stg_predictions


,column_name,column_type
0,station_id,VARCHAR
1,pulled_at_utc,TIMESTAMP
2,tide_at_utc,TIMESTAMP
3,height_ft_mllw,DOUBLE
4,tide_type,VARCHAR



stg_datums


,column_name,column_type
0,station_id,VARCHAR
1,units,VARCHAR
2,datum_epoch,VARCHAR
3,control_station,VARCHAR
4,mhhw_ft_stnd,DOUBLE
5,mllw_ft_stnd,DOUBLE
6,mhhw_above_mllw_ft,DOUBLE



stg_stations


,column_name,column_type
0,station_id,VARCHAR
1,station_name,VARCHAR
2,tide_pattern,VARCHAR
3,latitude,DOUBLE
4,longitude,DOUBLE


## 5. Mart: `mart_high_tides`

This table joins the three staging tables, keeps only high tides, and adds:

- `tide_at_local`: the tide time in Florida, for the alert message.
- `above_mhhw_ft`: the height minus the 1.56-foot gap, so 0 means a normal high tide.
- `crosses_threshold`: true when `above_mhhw_ft` is 0.6 or more.

In [17]:
show_file("sql/mart/mart_high_tides.sql")

create or replace table mart_high_tides as

with settings as (
    select 0.6 as margin_ft
),

highs as (
    select
        p.station_id,
        s.station_name,
        p.tide_at_utc,
        (p.tide_at_utc at time zone 'UTC') at time zone 'America/New_York' as tide_at_local,
        p.height_ft_mllw,
        round(p.height_ft_mllw - d.mhhw_above_mllw_ft, 3) as above_mhhw_ft
    from stg_predictions p
    join stg_datums d using (station_id)
    join stg_stations s using (station_id)
    where p.tide_type = 'high'
)

select
    h.*,
    cast(h.tide_at_local as date) as tide_date_local,
    cast(h.tide_at_local as date) - 3 as alert_date_local,
    h.above_mhhw_ft >= st.margin_ft as crosses_threshold
from highs h
cross join settings st


In [18]:
run_file("sql/mart/mart_high_tides.sql")
q("select * from mart_high_tides order by tide_at_utc limit 10")

Ran sql/mart/mart_high_tides.sql


,station_id,station_name,tide_at_utc,tide_at_local,height_ft_mllw,above_mhhw_ft,tide_date_local,alert_date_local,crosses_threshold
0,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-01 08:07:00,2026-08-01 04:07:00,1.144,-0.416,2026-08-01,2026-07-29,False
1,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-01 18:21:00,2026-08-01 14:21:00,1.621,0.061,2026-08-01,2026-07-29,False
2,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-02 08:23:00,2026-08-02 04:23:00,1.225,-0.335,2026-08-02,2026-07-30,False
3,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-02 19:09:00,2026-08-02 15:09:00,1.420,-0.140,2026-08-02,2026-07-30,False
4,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-03 08:32:00,2026-08-03 04:32:00,1.337,-0.223,2026-08-03,2026-07-31,False
5,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-03 20:19:00,2026-08-03 16:19:00,1.191,-0.369,2026-08-03,2026-07-31,False
6,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-04 08:36:00,2026-08-04 04:36:00,1.483,-0.077,2026-08-04,2026-08-01,False
7,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-04 22:28:00,2026-08-04 18:28:00,0.985,-0.575,2026-08-04,2026-08-01,False
8,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-05 08:57:00,2026-08-05 04:57:00,1.644,0.084,2026-08-05,2026-08-02,False
9,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-06 09:37:00,2026-08-06 05:37:00,1.791,0.231,2026-08-06,2026-08-03,False


In [19]:
q("""
    select crosses_threshold, count(*) as high_tides
    from mart_high_tides
    group by crosses_threshold
""")

,crosses_threshold,high_tides
0,False,536
1,True,14


The late-November king tide. Every crossing tide in November, plus the tides on either side, so you can see the 26th has no high tide at all:

In [20]:
q("""
    select tide_at_utc, tide_at_local, tide_date_local, height_ft_mllw, above_mhhw_ft, crosses_threshold
    from mart_high_tides
    where tide_date_local between '2026-11-23' and '2026-11-28'
    order by tide_at_utc
""")

,tide_at_utc,tide_at_local,tide_date_local,height_ft_mllw,above_mhhw_ft,crosses_threshold
0,2026-11-23 18:11:00,2026-11-23 13:11:00,2026-11-23,1.192,-0.368,False
1,2026-11-24 03:21:00,2026-11-23 22:21:00,2026-11-23,2.135,0.575,False
2,2026-11-25 03:57:00,2026-11-24 22:57:00,2026-11-24,2.246,0.686,True
3,2026-11-26 04:39:00,2026-11-25 23:39:00,2026-11-25,2.286,0.726,True
4,2026-11-27 05:24:00,2026-11-27 00:24:00,2026-11-27,2.240,0.680,True
5,2026-11-28 06:14:00,2026-11-28 01:14:00,2026-11-28,2.104,0.544,False


| Column | Value | Measured from | Reads as |
|---|---|---|---|
| `height_ft_mllw` | 2.286 | Average low tide | 2.286 feet above a normal low tide |
| `above_mhhw_ft` | 0.726 | Average high tide | 0.726 feet above a normal high tide |

## 6. Mart: `mart_alerts`

This is the hardest query in the project, so the next cells rebuild it one CTE at a time. Each view is filtered to October and November when displayed, so you can watch the November 24, 25, and 27 days become one event.

### Step 1: one row per crossing day

In [21]:
con.sql("""
    create or replace temp view peek_crossing_days as
    select
        station_id,
        station_name,
        tide_date_local,
        max(above_mhhw_ft) as peak_above_mhhw_ft,
        arg_max(tide_at_local, above_mhhw_ft) as peak_tide_at_local
    from mart_high_tides
    where crosses_threshold
    group by station_id, station_name, tide_date_local
""")

q("""
    select tide_date_local, peak_above_mhhw_ft, peak_tide_at_local
    from peek_crossing_days
    where tide_date_local between '2026-10-01' and '2026-11-30'
    order by tide_date_local
""")

,tide_date_local,peak_above_mhhw_ft,peak_tide_at_local
0,2026-10-01,0.609,2026-10-01 02:32:00
1,2026-10-02,0.600,2026-10-02 03:19:00
2,2026-10-27,0.603,2026-10-27 00:22:00
3,2026-10-28,0.706,2026-10-28 00:52:00
4,2026-10-29,0.742,2026-10-29 01:29:00
5,2026-10-30,0.700,2026-10-30 02:13:00
6,2026-11-24,0.686,2026-11-24 22:57:00
7,2026-11-25,0.726,2026-11-25 23:39:00
8,2026-11-27,0.680,2026-11-27 00:24:00


### Step 2: days since the previous crossing day

`lag()` reads the previous row's date. The very first crossing day of the year has nothing before it, so it gets null (shown as `NaN` in pandas).

In [22]:
con.sql("""
    create or replace temp view peek_gaps as
    select
        *,
        tide_date_local - lag(tide_date_local) over (
            partition by station_id order by tide_date_local
        ) as days_since_previous
    from peek_crossing_days
""")

q("""
    select tide_date_local, days_since_previous
    from peek_gaps
    where tide_date_local between '2026-10-01' and '2026-11-30'
    order by tide_date_local
""")

,tide_date_local,days_since_previous
0,2026-10-01,51
1,2026-10-02,1
2,2026-10-27,25
3,2026-10-28,1
4,2026-10-29,1
5,2026-10-30,1
6,2026-11-24,25
7,2026-11-25,1
8,2026-11-27,2


### Step 3: mark the days that start a new event

With `max_gap_days = 1`, a new event starts only when the gap is more than 2 days. November 27 has a gap of 2 (one quiet day, the 26th), so it gets 0 and stays in the same event.

In [23]:
con.sql("""
    create or replace temp view peek_flagged as
    select
        *,
        case
            when days_since_previous is null then 1
            -- 1 + 1 stands in for max_gap_days + 1 from the settings CTE
            when days_since_previous > 1 + 1 then 1
            else 0
        end as starts_event
    from peek_gaps
""")

q("""
    select tide_date_local, days_since_previous, starts_event
    from peek_flagged
    where tide_date_local between '2026-10-01' and '2026-11-30'
    order by tide_date_local
""")

,tide_date_local,days_since_previous,starts_event
0,2026-10-01,51,1
1,2026-10-02,1,0
2,2026-10-27,25,1
3,2026-10-28,1,0
4,2026-10-29,1,0
5,2026-10-30,1,0
6,2026-11-24,25,1
7,2026-11-25,1,0
8,2026-11-27,2,0


### Step 4: number the events with a running total

`sum(starts_event) over (...)` adds up the 1s as it goes down the rows. It goes up by 1 at each event start and stays flat inside an event, so every day in an event shares a number.

In [24]:
con.sql("""
    create or replace temp view peek_numbered as
    select
        *,
        sum(starts_event) over (
            partition by station_id order by tide_date_local
        ) as event_number
    from peek_flagged
""")

q("""
    select tide_date_local, days_since_previous, starts_event,
           cast(event_number as integer) as event_number
    from peek_numbered
    order by tide_date_local
""")

,tide_date_local,days_since_previous,starts_event,event_number
0,2026-08-10,<NA>,1,1
1,2026-08-11,1,0,1
2,2026-10-01,51,1,2
3,2026-10-02,1,0,2
4,2026-10-27,25,1,3
5,2026-10-28,1,0,3
6,2026-10-29,1,0,3
7,2026-10-30,1,0,3
8,2026-11-24,25,1,4
9,2026-11-25,1,0,4


### Steps 5 and 6: collapse each event and add the send date

Run the real file. Each event number from the cell above becomes one row.

In [25]:
show_file("sql/mart/mart_alerts.sql")

-- mart_alerts.sql
-- Mart table: one row per king tide event at each station.
-- An event is a run of days with at least one high tide over the threshold.
-- Each event gets one alert, sent a set number of days before it starts.
-- Reads mart_high_tides, which flags each tide that crosses the threshold.
-- "create or replace" means each run rebuilds this table from scratch,
-- so running it twice never adds rows.

create or replace table mart_alerts as

-- The alert rules, kept in one place so they're easy to change.
with settings as (
    select
        -- How many quiet days can sit inside one event. With 1, crossing days
        -- on November 24, 25, and 27 are one event, not two, so boat owners
        -- get one alert instead of a second one a day later.
        -- analysis/threshold_calibration.sql uses the same rule.
        1 as max_gap_days,
        -- How many days before an event's first day the alert goes out,
        -- so boat owners have time to secure their boats and 

In [26]:
run_file("sql/mart/mart_alerts.sql")
q("select * from mart_alerts")

Ran sql/mart/mart_alerts.sql


,alert_id,station_id,station_name,first_day,last_day,crossing_days,span_days,peak_tide_at_local,peak_above_mhhw_ft,send_date
0,8725577_20260810,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-08-10,2026-08-11,2,2,2026-08-11 11:39:00,0.646,2026-08-07
1,8725577_20261001,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-10-01,2026-10-02,2,2,2026-10-01 02:32:00,0.609,2026-09-28
2,8725577_20261027,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-10-27,2026-10-30,4,4,2026-10-29 01:29:00,0.742,2026-10-24
3,8725577_20261124,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2026-11-24,2026-11-27,3,4,2026-11-25 23:39:00,0.726,2026-11-21
4,8725577_20270703,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2027-07-03,2027-07-04,2,2,2027-07-04 12:28:00,0.647,2027-06-30
5,8725577_20270731,8725577,"PORT BOCA GRANDE, CHARLOTTE HBR.",2027-07-31,2027-07-31,1,1,2027-07-31 10:46:00,0.616,2027-07-28


> **Explain it back.** Why does the alert go out three days before `first_day` and not three days before the peak?

## 7. Tests

Each test is a query that returns rows **only when something is wrong**, so an empty table means it passed. This cell runs every file in `sql/tests/`, the same loop `transform.py` uses, and shows what each one returned.

In [27]:
for path in sorted(Path("sql/tests").glob("*.sql")):
    result = q(path.read_text(encoding="utf-8"))
    status = "PASS" if result.empty else "FAIL"
    print(f"\n{status} {path.name} ({len(result)} rows)")
    display(result)


PASS alerts_unique.sql (0 rows)


,alert_id,copies



PASS datums_in_feet.sql (0 rows)


,station_id,units,mhhw_above_mllw_ft



PASS predictions_not_null.sql (0 rows)


,station_id,pulled_at_utc,tide_at_utc,height_ft_mllw,tide_type


... (2 outputs left out)


PASS stations_have_datums.sql (0 rows)


,station_id



PASS stations_not_empty.sql (0 rows)


,problem


### Break a test on purpose

An empty table is hard to trust until you've seen a test fail. This cell adds a copy of one tide to `stg_predictions` and runs the uniqueness test again.

In [28]:
con.sql("""
    insert into stg_predictions
    select * from stg_predictions order by tide_at_utc limit 1
""")

q(Path("sql/tests/predictions_unique.sql").read_text(encoding="utf-8"))

,station_id,tide_at_utc,copies
0,8725577,2026-08-01 01:14:00,2


The test caught it. Now rebuild the table from its SQL file. Because the file starts with `create or replace`, the bad row is gone and the test passes again. That's the full-rebuild design doing its job: you never fix a table by hand, you rebuild it from the raw files.

In [29]:
run_file("sql/staging/stg_predictions.sql")
q(Path("sql/tests/predictions_unique.sql").read_text(encoding="utf-8"))

Ran sql/staging/stg_predictions.sql


,station_id,tide_at_utc,copies


## 8. What happens on a second run

Count the rows in every table, run all five SQL files again in `transform.py`'s order, and count again. The counts should match, because each file replaces its table instead of adding to it.

In [30]:
SQL_FILES = [
    "sql/staging/stg_predictions.sql",
    "sql/staging/stg_datums.sql",
    "sql/staging/stg_stations.sql",
    "sql/mart/mart_high_tides.sql",
    "sql/mart/mart_alerts.sql",
]
TABLES = ["stg_predictions", "stg_datums", "stg_stations", "mart_high_tides", "mart_alerts"]

def row_counts():
    return {t: con.sql(f"select count(*) from {t}").fetchall()[0][0] for t in TABLES}

before = row_counts()
for path in SQL_FILES:
    run_file(path)
after = row_counts()

pd.DataFrame({"first run": before, "second run": after})

Ran sql/staging/stg_predictions.sql
Ran sql/staging/stg_datums.sql
Ran sql/staging/stg_stations.sql
Ran sql/mart/mart_high_tides.sql
Ran sql/mart/mart_alerts.sql


,first run,second run
stg_predictions,1101,1101
stg_datums,1,1
stg_stations,1,1
mart_high_tides,550,550
mart_alerts,6,6


> **Explain it back.** What would change in these counts if the files used `insert into` instead of `create or replace`?

## 9. Output: the alert messages

`alerts.py` turns each row of `mart_alerts` into a message. This cell imports its `message()` function instead of copying it. The import works without sending anything because `alerts.py` only runs `main()` under `if __name__ == "__main__"`.

In [31]:
from alerts import message

alerts = q("select * from mart_alerts order by send_date")
alerts["message"] = [
    message(row.first_day, row.last_day, row.peak_tide_at_local, row.peak_above_mhhw_ft)
    for row in alerts.itertuples()
]

pd.set_option("display.max_colwidth", None)
alerts[["send_date", "first_day", "last_day", "message"]]

,send_date,first_day,last_day,message
0,2026-08-07,2026-08-10,2026-08-11,"King tide alert for Port Boca Grande: high water is predicted from Monday, August 10 to Tuesday, August 11. The highest tide is expected Tuesday, August 11 at 11:39 AM, about 8 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."
1,2026-09-28,2026-10-01,2026-10-02,"King tide alert for Port Boca Grande: high water is predicted from Thursday, October 1 to Friday, October 2. The highest tide is expected Thursday, October 1 at 2:32 AM, about 7 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."
2,2026-10-24,2026-10-27,2026-10-30,"King tide alert for Port Boca Grande: high water is predicted from Tuesday, October 27 to Friday, October 30. The highest tide is expected Thursday, October 29 at 1:29 AM, about 9 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."
3,2026-11-21,2026-11-24,2026-11-27,"King tide alert for Port Boca Grande: high water is predicted from Tuesday, November 24 to Friday, November 27. The highest tide is expected Wednesday, November 25 at 11:39 PM, about 9 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."
4,2027-06-30,2027-07-03,2027-07-04,"King tide alert for Port Boca Grande: high water is predicted from Saturday, July 3 to Sunday, July 4. The highest tide is expected Sunday, July 4 at 12:28 PM, about 8 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."
5,2027-07-28,2027-07-31,2027-07-31,"King tide alert for Port Boca Grande: high water is predicted on Saturday, July 31. The highest tide is expected Saturday, July 31 at 10:46 AM, about 7 inches above a normal high tide. Add slack to your dock lines and move anything stored on the dock. Wind can push the water higher than predicted."


## 10. Threshold calibration

The analysis query behind the 0.6-foot choice. It isn't part of the pipeline, but it reads `mart_high_tides`, so it needs the marts built first. The output should match the table in the README.

In [32]:
q(Path("analysis/threshold_calibration.sql").read_text(encoding="utf-8"))

,margin_ft,events,days_over_line
0,0.50,9.0,32
1,0.55,8.0,24
2,0.60,6.0,14
3,0.65,2.0,6
4,0.70,2.0,4


## Done

You've now seen every table the pipeline builds. To restart from a clean state, use **Kernel > Restart Kernel and Run All Cells**. Since the database lives in memory, a restart wipes it and every cell rebuilds it from the raw files.